# FluxCompute Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fluxcompute/fluxcompute-sdk/blob/main/examples/quickstart.ipynb)

**FluxCompute routes every query to the cheapest model that can answer it.** Queries that don't need the top tier cost 40–80% less; queries that genuinely do route to the frontier model and save nothing — so your overall reduction depends on your traffic mix. You'll measure it on a realistic mix below. Swap the client, pass `model="auto"`, and it picks Haiku / Sonnet / Opus per query and shows you exactly what you saved.

In ~5 minutes you'll: make your first auto-routed call, see *why* it picked a model, and measure the savings across a batch.

**Prerequisite:** an Anthropic API key ([console.anthropic.com](https://console.anthropic.com/)). Calls are real and cost a few cents total.

## 1 · Install

In [ ]:
%pip install -q "fluxcompute>=0.3.0" matplotlib pandas

## 2 · Authenticate & create the client

In [ ]:
import os, getpass

# FluxCompute calls Anthropic for real, so you need an Anthropic API key.
# Get one at https://console.anthropic.com/ . It is read via getpass — never
# hardcode keys in a notebook you might share.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (sk-ant-...): ")

from fluxcompute import FluxClient

# Drop-in replacement for anthropic.AsyncAnthropic. `baseline_model` defaults to
# the priciest tier (Opus) — that's what your savings are measured against.
client = FluxClient(anthropic_key=os.environ["ANTHROPIC_API_KEY"])
print("FluxClient ready.")

## 3 · Your first auto-routed call

The only change from the Anthropic SDK is `model="auto"`. FluxCompute classifies the query and routes it — here, an easy factual question goes to Haiku instead of Opus.

In [ ]:
# NOTE: the SDK is async. In a Jupyter cell you can `await` directly;
# in a plain .py script, wrap calls in asyncio.run(...).
resp = await client.messages.create(
    model="auto",                                   # let FluxCompute pick the model
    max_tokens=512,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)

print(resp.text, "\n")

m = resp.fluxcompute
pct = (m.savings_usd / m.baseline_cost_usd * 100) if m.baseline_cost_usd else 0
print(f"Routed to : {m.model_selected}  (difficulty: {m.difficulty_label}, score {m.difficulty_score})")
print(f"You paid  : ${m.cost_usd:.6f}")
print(f"All-Opus  : ${m.baseline_cost_usd:.6f}  (baseline: {m.baseline_model})")
print(f"Saved     : ${m.savings_usd:.6f}  ({pct:.0f}% cheaper) — for a question Opus is overkill for")

## 4 · Why it works — the routing decision

Send an easy, a medium, and a hard query. Watch `difficulty` → `routed_to` change, and compare each cost to what all-Opus would have cost.

In [ ]:
import pandas as pd

samples = [
    "What is 15% of 200?",                                                           # easy
    "Explain the difference between supervised and unsupervised learning.",          # medium
    ("Design a distributed rate limiter across 50 instances with Redis. Analyze "    # hard
     "the trade-offs between token bucket and sliding window, justify your choice, "
     "and prove the algorithm is correct under network partition."),
]

def short_model(name: str) -> str:
    """Drop the vendor prefix and any trailing date stamp for a readable table."""
    name = name.replace("claude-", "")
    parts = name.rsplit("-", 1)
    # only an 8-digit date stamp is a suffix; "-6" in sonnet-4-6 is part of
    # the version and stripping it renames the model
    is_date = len(parts) == 2 and len(parts[1]) == 8 and parts[1].isdigit()
    return parts[0] if is_date else name

rows = []
for q in samples:
    r = await client.messages.create(model="auto", max_tokens=512,
                                      messages=[{"role": "user", "content": q}])
    m = r.fluxcompute
    rows.append({
        "query": (q[:45] + "\u2026") if len(q) > 45 else q,
        "difficulty": m.difficulty_label,
        "routed_to": short_model(m.model_selected),
        "cost_usd": round(m.cost_usd, 6),
        "all_opus_usd": round(m.baseline_cost_usd, 6),
        "saved_usd": round(m.savings_usd, 6),
    })

pd.DataFrame(rows)

## 5 · The savings, in one number

Across a realistic mix of traffic, the savings compound. This is the whole pitch in one chart.

In [ ]:
import matplotlib.pyplot as plt

# A realistic mix: mostly easy/medium traffic, with a couple of genuinely hard asks.
bank = [
    "What does HTTP stand for?",
    "Translate 'good morning' to Japanese.",
    "Who wrote Pride and Prejudice?",
    "What is 240 / 8?",
    "Summarize the CAP theorem in two sentences.",
    "Compare REST and GraphQL and say when to use each.",
    "Write a Python function that checks if a string is a palindrome.",
    "Explain how a transformer attention head works.",
    ("Implement an LFU cache with O(1) get/put in Python. Analyze the trade-offs "
     "against an LRU design, prove the complexity bound, and justify the tie-breaking rule."),
    ("Debug why adding a composite index to a 500M-row Postgres table under 50k "
     "inserts/sec deadlocks. Analyze the lock ordering, prove the fix removes the "
     "cycle, and evaluate the trade-offs of doing it online."),
]

metas = []
for q in bank:
    r = await client.messages.create(model="auto", max_tokens=512,
                                      messages=[{"role": "user", "content": q}])
    metas.append(r.fluxcompute)

total_actual   = sum(m.cost_usd for m in metas)
total_baseline = sum(m.baseline_cost_usd for m in metas)
total_saved    = sum(m.savings_usd for m in metas)
pct = (total_saved / total_baseline * 100) if total_baseline else 0

print(f"{len(metas)} queries")
print(f"All-Opus baseline : ${total_baseline:.4f}")
print(f"FluxCompute (auto): ${total_actual:.4f}")
print(f"Saved             : ${total_saved:.4f}   ({pct:.0f}% reduction)")
print()
print("Below the 80%/40% per-query figures because the hard queries route to")
print("Opus — the baseline itself — so they save nothing by construction.")
print("Your number depends on how much of your traffic needs a frontier model.")

plt.figure(figsize=(5, 3.2))
bars = plt.bar(["All-Opus\n(baseline)", "FluxCompute\n(auto)"],
               [total_baseline, total_actual], color=["#cbd5e1", "#3b82f6"])
plt.ylabel("Total cost (USD)")
plt.title(f"{pct:.0f}% cheaper on this workload")
for b, v in zip(bars, [total_baseline, total_actual]):
    plt.text(b.get_x() + b.get_width() / 2, v, f"${v:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 6 · It's a drop-in replacement

`FluxResponse` exposes `.text`, `.usage`, and `.model` just like the raw response, so migrating is a two-line change.

In [ ]:
# FluxResponse behaves like the raw provider response, so your existing code keeps working:
print("model :", resp.model)
print("usage :", resp.usage)            # {'input_tokens': ..., 'output_tokens': ...}
print("text  :", resp.text[:60], "...")

# Migrating from the Anthropic SDK — swap the import and client,
# then pass model="auto":
#
#   - from anthropic import AsyncAnthropic
#   - client = AsyncAnthropic(api_key=KEY)
#   - resp = await client.messages.create(model="claude-opus-4-8", max_tokens=512, messages=msgs)
#   - text = resp.content[0].text
#   + from fluxcompute import FluxClient
#   + client = FluxClient(anthropic_key=KEY)
#   + resp = await client.messages.create(model="auto", max_tokens=512, messages=msgs)
#   + text = resp.text          # resp.content / resp.usage / resp.model also work

## Next steps

- **Multi-turn sessions, streaming & execution graphs** → see [`full_walkthrough.ipynb`](full_walkthrough.ipynb)
- **Cloud dashboard** (invite-only early access) → pass `fluxcompute_key="flx_..."` to `FluxClient(...)` to stream routing and cost metrics. Response text stays on your machine unless you also pass `content_capture=True` — see the Telemetry & privacy section of the [README](../README.md). Without a reachable endpoint this is a silent no-op; request access at [fluxcompute.dev](https://fluxcompute.dev).
- **Resuming a failed step** → `client.resume()` is built into the SDK and free — see the demo in [`full_walkthrough.ipynb`](full_walkthrough.ipynb). Durable, cross-process recovery (resuming a task after the process that ran it has exited) is part of the hosted platform ([fluxcompute.dev](https://fluxcompute.dev))

Clean up when you're done:

In [ ]:
# Flushes telemetry and closes the underlying provider client.
await client.close()
print("done.")